In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import nltk

In [3]:
df=pd.read_csv("/content/nlp_dataset.csv")
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [4]:
df.shape

(5937, 2)

In [5]:
print(df.columns)

Index(['Comment', 'Emotion'], dtype='object')


In [7]:
print(df.isnull().sum())
print(df["Emotion"].value_counts())

Comment    0
Emotion    0
dtype: int64
Emotion
anger    2000
joy      2000
fear     1937
Name: count, dtype: int64


In [11]:
df["Emotion"].unique()

array(['fear', 'anger', 'joy'], dtype=object)

In [12]:
import re
def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+|www\S+", "", text)
    text=re.sub(r"[^a-zA-Z\s']", " ", text)
    text=re.sub(r"\s+", " ", text)
    return text.strip()
df = df.dropna(subset=["Comment", "Emotion"]).copy()
df["Comment"] = df["Comment"].astype(str)
df["Comment"] = df["Comment"].apply(clean_text)
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


**tokenization and stop word removal**

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
stop_words = stop_words-{"not", "no", "nor", "never"}

def preprocess_text(text):
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["Comment"] = df["Comment"].apply(preprocess_text)
df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Comment,Emotion
0,seriously hate one subject death feel reluctan...,fear
1,im full life feel appalled,anger
2,sit write start dig feelings think afraid acce...,fear
3,ive really angry r feel like idiot trusting fi...,joy
4,feel suspicious no one outside like rapture ha...,fear


**separating input and output**

In [14]:
X = df["Comment"]
y = df["Emotion"]

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(X, y,test_size=0.30,random_state=42,stratify=y)
print(len(X_train))
print(len(X_test))

4155
1782


**converting text into numerical features**

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=20000)
X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)

In [20]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(4155, 7144)
(1782, 7144)


**naive bayes model**

In [21]:
from sklearn.naive_bayes import MultinomialNB
nb=MultinomialNB()
nb.fit(X_train_tfidf, y_train)
nb_pred=nb.predict(X_test_tfidf)

In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [24]:
nb_accuracy=accuracy_score(y_test,nb_pred)
nb_precision=precision_score(y_test,nb_pred, average="weighted", zero_division=0)
nb_recall=recall_score(y_test,nb_pred, average="weighted", zero_division=0)
nb_f1=f1_score(y_test, nb_pred,average="weighted", zero_division=0)

In [25]:
print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes Precision:", nb_precision)
print("Naive Bayes Recall:", nb_recall)
print("Naive Bayes F1 Score:", nb_f1)

Naive Bayes Accuracy: 0.8916947250280584
Naive Bayes Precision: 0.8932082804640983
Naive Bayes Recall: 0.8916947250280584
Naive Bayes F1 Score: 0.8917450428422087


**svm model**

In [22]:
from sklearn.svm import LinearSVC
svm=LinearSVC(random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_pred=svm.predict(X_test_tfidf)

In [26]:
svm_accuracy=accuracy_score(y_test,svm_pred)
svm_precision=precision_score(y_test,svm_pred, average="weighted", zero_division=0)
svm_recall=recall_score(y_test,svm_pred, average="weighted", zero_division=0)
svm_f1=f1_score(y_test,svm_pred, average="weighted", zero_division=0)

In [27]:
print("SVM Accuracy:", svm_accuracy)
print("SVM Precision:", svm_precision)
print("SVM Recall:", svm_recall)
print("SVM F1 Score:", svm_f1)

SVM Accuracy: 0.9332210998877666
SVM Precision: 0.9332379803470913
SVM Recall: 0.9332210998877666
SVM F1 Score: 0.9332264468228053


**comparing**

In [28]:
results = pd.DataFrame({
    "Model": ["Naive Bayes", "SVM"],
    "Accuracy": [nb_accuracy, svm_accuracy],
    "Precision": [nb_precision, svm_precision],
    "Recall": [nb_recall, svm_recall],
    "F1-score": [nb_f1, svm_f1]
})

print(results.round(4))

         Model  Accuracy  Precision  Recall  F1-score
0  Naive Bayes    0.8917     0.8932  0.8917    0.8917
1          SVM    0.9332     0.9332  0.9332    0.9332


In [30]:
#classification report
from sklearn.metrics import classification_report
print("NAIVE BAYES REPORT")
print(classification_report(y_test, nb_pred, zero_division=0))
print("SVM REPORT")
print(classification_report(y_test, svm_pred, zero_division=0))

NAIVE BAYES REPORT
              precision    recall  f1-score   support

       anger       0.87      0.91      0.89       600
        fear       0.88      0.90      0.89       582
         joy       0.93      0.86      0.89       600

    accuracy                           0.89      1782
   macro avg       0.89      0.89      0.89      1782
weighted avg       0.89      0.89      0.89      1782

SVM REPORT
              precision    recall  f1-score   support

       anger       0.93      0.93      0.93       600
        fear       0.92      0.93      0.93       582
         joy       0.94      0.94      0.94       600

    accuracy                           0.93      1782
   macro avg       0.93      0.93      0.93      1782
weighted avg       0.93      0.93      0.93      1782



**predicting**

In [32]:
new_comment=["i do not always find myself feeling thankful but over the years i ve gathered a few tricks that allow me to feel grateful in the face of moments when the last thing i want to do is say thanks"]
cleaned_comment=[preprocess_text(clean_text(new_comment[0]))]
comment_tfidf=vectorizer.transform(cleaned_comment)
prediction=svm.predict(comment_tfidf)
print("Comment:", new_comment[0])
print("Predicted Emotion:", prediction[0])

Comment: i do not always find myself feeling thankful but over the years i ve gathered a few tricks that allow me to feel grateful in the face of moments when the last thing i want to do is say thanks
Predicted Emotion: joy
